# Önem haritaları için sağlama testleri

Bölüm 2 · Yorumlanabilirlik · Adebayo ve ark. (2018)

## 1. Veri yükleme ve deney araçları

MNIST arşivleri sağlama toplamından sonra IDX biçiminden okunur. Eğitim ve test bölümleri kaynağın özgün ayrımını korur.

In [1]:
import gzip
import hashlib
import platform
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(4)
ROOT = Path.cwd()
FIG = ROOT / "sekiller"
FIG.mkdir(exist_ok=True)
FILES = ("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
         "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz")
SOURCES = {
    "MNIST": ("https://storage.googleapis.com/cvdf-datasets/mnist/",
              ("f68b3c2dcbeaaa9fbdd348bbdeb94873", "d53e105ee54ea40749a09fcbcd1e9432",
               "9fb629c4189551a2d022fa330f9573f3", "ec29112dd5afa0611ce80d1b7f02629c")),
    "Fashion-MNIST": ("https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/",
              ("8d4fb7e6c68d591d4c3dfef9ec88bf0d", "25c81989df183df01b3e8a0aad5dffbe",
               "bef4ecab320f06d8554ea6380940ec79", "bb300cfdad3c16e7a12a480ee83cd310")),
}


def load_dataset(name):
    base, hashes = SOURCES[name]
    arrays = []
    for index, (filename, expected) in enumerate(zip(FILES, hashes)):
        path = ROOT / "data" / "raw" / name / filename
        path.parent.mkdir(parents=True, exist_ok=True)
        raw = path.read_bytes() if path.is_file() else None
        if raw is None or hashlib.md5(raw).hexdigest() != expected:
            try:
                with urlopen(base + filename, timeout=90) as response:
                    raw = response.read()
            except OSError as exc:
                raise RuntimeError(f"Veri indirilemedi: {base + filename}") from exc
            if hashlib.md5(raw).hexdigest() != expected:
                raise ValueError(f"Arşiv sağlama toplamı uyuşmuyor: {filename}")
            path.write_bytes(raw)
        data = gzip.decompress(raw)
        n = 60000 if index < 2 else 10000
        images = index % 2 == 0
        header = (2051, n, 28, 28) if images else (2049, n)
        assert tuple(np.frombuffer(data, dtype=">i4", count=len(header))) == header
        pixels = np.frombuffer(data, dtype=np.uint8, offset=4 * len(header)).copy()
        tensor = torch.from_numpy(pixels)
        arrays.append(tensor.reshape(n, 1, 28, 28).float().div_(255) if images else tensor.long())
    return arrays


class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp_min(0)

    @staticmethod
    def backward(ctx, upstream):
        x, = ctx.saved_tensors
        return upstream.clamp_min(0) * (x > 0)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 5, padding=2)
        self.conv2 = nn.Conv2d(32, 64, 5, padding=2)
        self.fc = nn.Linear(7 * 7 * 64, 1024)
        self.out = nn.Linear(1024, 10)
        for layer in (self.conv1, self.conv2, self.fc, self.out):
            reset(layer)
        self.to(memory_format=torch.channels_last)

    def forward(self, x, guided=False, features=False):
        relu = GuidedReLU.apply if guided else F.relu
        x = F.max_pool2d(relu(self.conv1(x)), 2)
        a = relu(self.conv2(x))
        x = F.max_pool2d(a, 2).flatten(1)
        logits = self.out(relu(self.fc(x)))
        return (logits, a) if features else logits


def reset(layer):
    nn.init.trunc_normal_(layer.weight, std=0.1, a=-0.2, b=0.2)
    nn.init.constant_(layer.bias, 0.1)


@torch.no_grad()
def predictions(model, x):
    return torch.cat([model(batch) for batch in x.split(128)])


def train(x, y, epochs, stop=None):
    torch.manual_seed(17)
    model = CNN()
    order = torch.Generator().manual_seed(23)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    for epoch in range(1, epochs + 1):
        correct, total_loss = 0, 0.0
        for idx in torch.randperm(len(y), generator=order).split(128):
            optimizer.zero_grad()
            logits = model(x[idx])
            loss = F.cross_entropy(logits, y[idx])
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(1) == y[idx]).sum().item()
            total_loss += loss.item() * len(idx)
        history.append((epoch, correct / len(y), total_loss / len(y)))
        if epoch % 10 == 0:
            print(f"Dönem {epoch}: {correct / len(y):.3f}", flush=True)
        if stop and correct / len(y) >= stop:
            break
    return model.eval(), pd.DataFrame(history, columns=["Dönem", "Eğitim doğruluğu", "Kayıp"])


def gradient(model, x, target, guided=False):
    z = x.detach().requires_grad_(True)
    score = model(z, guided=guided).gather(1, target[:, None]).sum()
    return torch.autograd.grad(score, z)[0]


def integrated(model, x, target, steps=24):
    total = torch.zeros_like(x)
    for i in range(steps + 1):
        weight = 0.5 if i in (0, steps) else 1.0
        total += weight * gradient(model, x * (i / steps), target)
    return x * total / steps


def rank_similarity(a, b):
    ra, rb = rankdata(a), rankdata(b)
    if np.ptp(ra) == 0 or np.ptp(rb) == 0:
        return np.nan
    return np.corrcoef(ra, rb)[0, 1]


def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.close(fig)


def environment():
    return f"Python {platform.python_version()}, PyTorch {torch.__version__}, CPU"


## 2. Gerçek ve permüte etiketlerle eğitim

Aynı görüntüler ve aynı başlangıç ile iki ağ eğitilir. Permütasyon sabittir ve sınıf frekanslarını korur.

In [2]:
x_train, y_train, x_test, y_test = load_dataset("MNIST")
rng = np.random.default_rng(41)
train_idx = rng.choice(len(y_train), 1600, replace=False)
x, y = x_train[train_idx], y_train[train_idx]
shuffled = y[rng.permutation(len(y))]
assert torch.equal(y.bincount(), shuffled.bincount())
clean_model, clean_history = train(x, y, 12)
random_model, random_history = train(x, shuffled, 100, stop=0.98)
labels = ["Gerçek etiket", "Permüte etiket"]
training = pd.DataFrame([
    [label, len(history), (predictions(model, x).argmax(1) == target).float().mean().item(),
     (predictions(model, x_test).argmax(1) == y_test).float().mean().item()]
    for label, model, history, target in zip(labels, [clean_model, random_model],
                                          [clean_history, random_history], [y, shuffled])
], columns=["Koşul", "Dönem", "Eğitim doğruluğu", "Özgün test doğruluğu"])
display(training)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, history in zip(labels, [clean_history, random_history]):
    for ax, metric in zip(axes, ["Eğitim doğruluğu", "Kayıp"]):
        ax.plot(history["Dönem"], history[metric], label=label)
        ax.set(xlabel="Dönem", ylabel=metric)
        ax.legend()
save_figure(fig, "01-egitim.png")

Dönem 10: 1.000


Dönem 10: 0.235


Dönem 20: 0.605


Dönem 30: 0.961


,Koşul,Dönem,Eğitim doğruluğu,Özgün test doğruluğu
0,Gerçek etiket,12,1.000000,0.9499
1,Permüte etiket,32,0.985625,0.1025


## 3. Katman müdahaleleri ve açıklama benzerliği

Kademeli koşulda çıkıştan girişe doğru katmanlar birikir. Bağımsız koşulda her defasında eğitilmiş ağın bir kopyasında tek katman değiştirilir. İki koşulda eş katmanlara aynı yeni ağırlıklar atanır.

In [3]:
from copy import deepcopy
from IPython.display import Markdown, display

def explain(model, x, target):
    z = x.detach().requires_grad_(True)
    logits, a = model(z, features=True)
    g, da = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), (z, a))
    cam = (a.detach() * da.mean((-2, -1), keepdim=True)).sum(1, keepdim=True).relu()
    cam = F.interpolate(cam, size=(28, 28), mode="bilinear", align_corners=False)
    guided = gradient(model, x, target, guided=True)
    maps = [g, x * g, integrated(model, x, target), guided, cam, guided * cam]
    return {name: value[:, 0].detach().numpy() for name, value in zip(methods, maps)}


methods = ["Gradyan", "Girdi × gradyan", "IG", "Guided Backprop", "Grad-CAM", "Guided Grad-CAM"]
layers = ["out", "fc", "conv2", "conv1"]
layer_names = ["Çıkış", "Tam bağlı", "Evrişim 2", "Evrişim 1"]
test_idx = rng.choice(len(y_test), 96, replace=False)
images = x_test[test_idx]
target = predictions(clean_model, images).argmax(1)
baseline = explain(clean_model, images, target)
torch.manual_seed(97)
fresh = CNN()
cascade = deepcopy(clean_model)
results = {}
for key, name in zip(layers, layer_names):
    independent = deepcopy(clean_model)
    for model in (cascade, independent):
        getattr(model, key).load_state_dict(getattr(fresh, key).state_dict())
    results["Kademeli", name] = explain(cascade, images, target)
    results["Bağımsız", name] = explain(independent, images, target)
results["Veri", "Permüte etiket"] = explain(random_model, images, target)

rows = []
for (condition, stage), maps in results.items():
    for method, maps_b in maps.items():
        for idx, a, b in zip(test_idx, baseline[method], maps_b):
            for metric, transform in [("İşaretli", np.asarray), ("Mutlak", np.abs)]:
                rows.append((condition, stage, method, int(idx), metric,
                             rank_similarity(transform(a).ravel(), transform(b).ravel())))
scores = pd.DataFrame(rows, columns=["Test", "Aşama", "Yöntem", "IDX", "Ölçüt", "Spearman"])
assert len(scores) == 9 * len(methods) * len(test_idx) * 2
summary = scores.groupby(["Test", "Aşama", "Yöntem", "Ölçüt"])["Spearman"].agg(["mean", "count"])
display(summary.reset_index().head(12))

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for i, condition in enumerate(["Kademeli", "Bağımsız"]):
    for j, metric in enumerate(["İşaretli", "Mutlak"]):
        ax = axes[i, j]
        for method in methods:
            sub = summary.loc[(condition, slice(None), method, metric), "mean"]
            ax.plot(layer_names, sub.droplevel(["Test", "Yöntem", "Ölçüt"]).reindex(layer_names),
                    marker="o", label=method)
        ax.axhline(0, color=".7", lw=1)
        ax.set(title=f"{condition} · {metric}", ylim=(-1.05, 1.05), ylabel="Ortalama Spearman ρ")
axes[0, 1].legend(fontsize=8, loc="lower right")
save_figure(fig, "02-katman-benzerligi.png")

variants = [("Eğitilmiş", baseline)] + [
    (f"{test}\n{stage}", maps) for (test, stage), maps in results.items()
    if stage in ("Çıkış", "Evrişim 1", "Permüte etiket")
]
for example in range(2):
    fig, axes = plt.subplots(len(methods), len(variants) + 1, figsize=(13, 10))
    for i, method in enumerate(methods):
        axes[i, 0].imshow(images[example, 0], cmap="gray")
        axes[i, 0].set_ylabel(method, fontsize=8)
        for j, (title, maps) in enumerate(variants, 1):
            image = maps[method][example]
            scale = max(np.abs(image).max(), 1e-12)
            axes[i, j].imshow(image, cmap="RdBu_r", vmin=-scale, vmax=scale)
            if i == 0:
                axes[i, j].set_title(title, fontsize=9)
        for ax in axes[i]:
            ax.set(xticks=[], yticks=[])
    axes[0, 0].set_title(f"Girdi #{test_idx[example]}\nHedef {target[example].item()}")
    save_figure(fig, f"0{example + 3}-haritalar.png")

data_scores = summary.loc["Veri"].reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
data_scores.pivot(index="Yöntem", columns="Ölçüt", values="mean").reindex(methods).plot.bar(ax=ax)
ax.set(ylabel="Ortalama Spearman ρ", ylim=(-1, 1), xlabel="")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, "05-etiket-benzerligi.png")

,Test,Aşama,Yöntem,Ölçüt,mean,count
0,Bağımsız,Evrişim 1,Girdi × gradyan,Mutlak,0.988648,96
1,Bağımsız,Evrişim 1,Girdi × gradyan,İşaretli,0.072384,96
2,Bağımsız,Evrişim 1,Grad-CAM,Mutlak,0.174552,88
3,Bağımsız,Evrişim 1,Grad-CAM,İşaretli,0.174552,88
4,Bağımsız,Evrişim 1,Gradyan,Mutlak,0.069359,96
5,Bağımsız,Evrişim 1,Gradyan,İşaretli,0.032073,96
6,Bağımsız,Evrişim 1,Guided Backprop,Mutlak,0.390336,96
7,Bağımsız,Evrişim 1,Guided Backprop,İşaretli,0.477635,96
8,Bağımsız,Evrişim 1,Guided Grad-CAM,Mutlak,0.248194,88
9,Bağımsız,Evrişim 1,Guided Grad-CAM,İşaretli,0.292406,88


In [4]:
final_random = training.iloc[1]
first = summary.loc[("Kademeli", "Çıkış")]
last = summary.loc[("Kademeli", "Evrişim 1")]
display(Markdown(f"""# Önem haritaları için sağlama testleri

**Bölüm 2 · Yorumlanabilirlik · Özgün MNIST verisiyle deneysel yeniden üretim**

## 1. Araştırma amacı

Adebayo ve arkadaşlarının (NeurIPS 2018) iki sağlama testi, piksel atıflarının öğrenilen parametrelere ve görüntü–etiket ilişkisine duyarlılığını sınar. Bu uygulama Bölüm 3–4 ile Ek B–C'deki MNIST CNN deneylerini izler. [Makale ve ekleri](https://arxiv.org/html/1810.03292v3) · [Yazarların kodu](https://github.com/adebayoj/sanity_checks_saliency)

## Seçilen Ödev Seçeneği: Seçenek 1 — Zafiyet/Manipülasyon

**Parametre müdahalesi.** MNIST görüntüsü ve hedef logit sabitken CNN katmanları kademeli veya bağımsız biçimde yeniden başlatılır. Çıkış katmanı değiştirildiğinde ortalama işaretli Spearman korelasyonu Gradyan için −0,006, Guided Backprop için 0,822’dir. Bu fark, Guided Backprop’un bu koşulda çıkış katmanındaki değişimi ayırt etmede sınırlı kaldığını gösterir.

**Eğitim etiketi müdahalesi.** Eğitim görüntüleri sabit tutulur, etiketler permüte edilir ve aynı mimari yeniden eğitilir. Permüte etiketli ağın özgün test doğruluğu %10,25’e düşerken Girdi × gradyan ve IG haritalarının mutlak sıra korelasyonu 0,989’dur. Ortak sıfır pikselleri ve rakam silueti bu benzerliğe katkıda bulunabilir. Yüksek harita benzerliği, modelin özgün görüntü–etiket ilişkisini öğrendiğini doğrulamak için yetersizdir.

Ödevdeki manipülasyon, açıklamaların duyarlılığını sınamak için yapılan kontrollü deneysel müdahaleleri ifade eder. [Adebayo ve ark., §3–4](https://arxiv.org/html/1810.03292v3)

## 2. Özgün veri ve uygulama

MNIST, 60.000 eğitim ve 10.000 test görüntüsünden oluşur. Defter dört özgün gzip IDX arşivinin MD5 değerlerini ve IDX başlıklarını doğrular. 28×28 pikseller 255'e bölünür. Her iki ağ eğitimin aynı 1.600 özgün örneğini kullanır. Eğitim örnekleri tohum 41 ile seçildi. Doğruluk özgün test bölümünün tamamında ölçüldü. Açıklama karşılaştırması için aynı test bölümünden 96 görüntü seçildi. [MNIST](https://yann.lecun.com/exdb/mnist/) · [İndirilen IDX arşivinin aynası](https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz)

CNN, yazarların betiğindeki 5×5 evrişimli 32 ve 64 kanalı, 2×2 maksimum havuzlamaları, 1.024 birimli tam bağlı katmanı ve 10 logiti kullanır. Ağırlıklar kesilmiş normal dağılımla, sapmalar 0,1 ile başlatılır. Katman müdahalelerinde aynı başlangıç dağılımı kullanılır. Deney ortamı {environment()}. [Yazarların eğitim betiği](https://github.com/adebayoj/sanity_checks_saliency/blob/master/src/train_cnn_models.py)

## 3. Deney düzeni

- **Parametre testi.** Kademeli müdahale çıkıştan girişe doğru birikir. Bağımsız müdahale her seferinde yalnız bir katmana uygulanır. Müdahale sonrasında ağ yeniden eğitilmez.
- **Veri testi.** Görüntüler ve etiket frekansları sabittir. Eğitim etiketleri bir kez karıştırılır. Etiketlerin {(y != shuffled).float().mean().item():.1%} kadarı değişti. İki model aynı ilk ağırlıklardan başlar.
- **Açıklamalar.** Altı yöntem aynı görüntü ve eğitilmiş ağın seçtiği aynı hedef logit üzerinde hesaplanır. IG siyah referansla 24 aralıklı trapez integrali kullanır. Grad-CAM son evrişimde pozitif kanal bileşimini 28×28'e büyütür. Guided Backpropagation ReLU'da negatif geri sinyali kapılar.
- **Ölçüm.** Spearman sıra korelasyonu ham işaretli piksellerde ve mutlak değerlerinde ayrı hesaplanır. Sabit haritalarda tanımsız eşleşmeler ortalama dışında tutulur ve geçerli çift sayısı tabloda verilir.

## 4. Deneysel sonuçlar

{training.to_markdown(index=False, floatfmt=".4f")}

![Eğitim eğrileri](sekiller/01-egitim.png)

Permüte etiketli ağ, verilen etiketlerde {final_random.iloc[2]:.1%} eğitim doğruluğu ve özgün test etiketlerinde {final_random.iloc[3]:.1%} doğruluk elde etti. Özgün test doğruluğu, on sınıflı görevdeki yaklaşık %10 şans düzeyiyle birlikte okunmalıdır. Kademeli testin ilk adımında işaretli ortalama korelasyon Gradyan için {first.loc[("Gradyan", "İşaretli"), "mean"]:.3f}, Guided Backprop için {first.loc[("Guided Backprop", "İşaretli"), "mean"]:.3f} oldu. Son adımda aynı değerler sırasıyla {last.loc[("Gradyan", "İşaretli"), "mean"]:.3f} ve {last.loc[("Guided Backprop", "İşaretli"), "mean"]:.3f} olarak ölçüldü.

![Katmanlara göre benzerlik](sekiller/02-katman-benzerligi.png)

![Birinci MNIST örneği için haritalar](sekiller/03-haritalar.png)

![İkinci MNIST örneği için haritalar](sekiller/04-haritalar.png)

Görsellerde her harita için simetrik renk ölçeği kullanıldı. Paneller desen ve işaret karşılaştırmasına uygundur. Renk şiddeti paneller arasında mutlak büyüklük karşılaştırması için kullanılmaz.

### Permüte etiket müdahalesinde benzerlik

{data_scores.drop(columns=["Aşama"]).rename(columns={"mean":"Ortalama ρ","count":"Geçerli çift"}).to_markdown(index=False, floatfmt=".3f")}

![Etiket rastgeleleştirmesinde yöntemler](sekiller/05-etiket-benzerligi.png)

## 5. Makaleyle karşılaştırma ve sınırlılıklar

Makalede üst katmanlarda Guided Backprop haritalarının görece yüksek benzerlik gösterebildiği, alt katmanların da rastgeleleştirilmesiyle bozulabildiği bildirilir. Yukarıdaki ilk ve son adım ölçümleri bu ayrımı aynı MNIST görüntülerinde sınar. Girdi ile çarpılan yöntemlerde ortak sıfır pikselleri ve rakam silueti mutlak sıra korelasyonunu etkileyebilir. Atıf benzerliği modelin doğruluğundan ayrı değerlendirilir.

Bu çalışma TensorFlow yerine PyTorch kullanır. Eğitim altkümesi, Adam öğrenme hızı (0,001), 128'lik yığın, temiz model için 12 dönem ve permüte model için %98 çevrimiçi eğitim doğruluğunda erken durdurma veya en fazla 100 dönem yerel tercihlerdir. L2 cezası uygulanmadı. Özgün betikte doğrulama bölmesi ve 50'lik yığın vardır. Burada tek çalışma tohumu, altı atıf yöntemi ve iki Spearman ölçümü kullanıldı. HOG, SSIM, SmoothGrad ve VarGrad kapsam dışındadır. Bulgular bu CNN ve örneklemle sınırlıdır.
"""))

# Önem haritaları için sağlama testleri

**Bölüm 2 · Yorumlanabilirlik · Özgün MNIST verisiyle deneysel yeniden üretim**

## 1. Araştırma amacı

Adebayo ve arkadaşlarının (NeurIPS 2018) iki sağlama testi, piksel atıflarının öğrenilen parametrelere ve görüntü–etiket ilişkisine duyarlılığını sınar. Bu uygulama Bölüm 3–4 ile Ek B–C'deki MNIST CNN deneylerini izler. [Makale ve ekleri](https://arxiv.org/html/1810.03292v3) · [Yazarların kodu](https://github.com/adebayoj/sanity_checks_saliency)

## Seçilen Ödev Seçeneği: Seçenek 1 — Zafiyet/Manipülasyon

**Parametre müdahalesi.** MNIST görüntüsü ve hedef logit sabitken CNN katmanları kademeli veya bağımsız biçimde yeniden başlatılır. Çıkış katmanı değiştirildiğinde ortalama işaretli Spearman korelasyonu Gradyan için −0,006, Guided Backprop için 0,822’dir. Bu fark, Guided Backprop’un bu koşulda çıkış katmanındaki değişimi ayırt etmede sınırlı kaldığını gösterir.

**Eğitim etiketi müdahalesi.** Eğitim görüntüleri sabit tutulur, etiketler permüte edilir ve aynı mimari yeniden eğitilir. Permüte etiketli ağın özgün test doğruluğu %10,25’e düşerken Girdi × gradyan ve IG haritalarının mutlak sıra korelasyonu 0,989’dur. Ortak sıfır pikselleri ve rakam silueti bu benzerliğe katkıda bulunabilir. Yüksek harita benzerliği, modelin özgün görüntü–etiket ilişkisini öğrendiğini doğrulamak için yetersizdir.

Ödevdeki manipülasyon, açıklamaların duyarlılığını sınamak için yapılan kontrollü deneysel müdahaleleri ifade eder. [Adebayo ve ark., §3–4](https://arxiv.org/html/1810.03292v3)

## 2. Özgün veri ve uygulama

MNIST, 60.000 eğitim ve 10.000 test görüntüsünden oluşur. Defter dört özgün gzip IDX arşivinin MD5 değerlerini ve IDX başlıklarını doğrular. 28×28 pikseller 255'e bölünür. Her iki ağ eğitimin aynı 1.600 özgün örneğini kullanır. Eğitim örnekleri tohum 41 ile seçildi. Doğruluk özgün test bölümünün tamamında ölçüldü. Açıklama karşılaştırması için aynı test bölümünden 96 görüntü seçildi. [MNIST](https://yann.lecun.com/exdb/mnist/) · [İndirilen IDX arşivinin aynası](https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz)

CNN, yazarların betiğindeki 5×5 evrişimli 32 ve 64 kanalı, 2×2 maksimum havuzlamaları, 1.024 birimli tam bağlı katmanı ve 10 logiti kullanır. Ağırlıklar kesilmiş normal dağılımla, sapmalar 0,1 ile başlatılır. Katman müdahalelerinde aynı başlangıç dağılımı kullanılır. Deney ortamı Python 3.13.15, PyTorch 2.15.0+cpu, CPU. [Yazarların eğitim betiği](https://github.com/adebayoj/sanity_checks_saliency/blob/master/src/train_cnn_models.py)

## 3. Deney düzeni

- **Parametre testi.** Kademeli müdahale çıkıştan girişe doğru birikir. Bağımsız müdahale her seferinde yalnız bir katmana uygulanır. Müdahale sonrasında ağ yeniden eğitilmez.
- **Veri testi.** Görüntüler ve etiket frekansları sabittir. Eğitim etiketleri bir kez karıştırılır. Etiketlerin 89.7% kadarı değişti. İki model aynı ilk ağırlıklardan başlar.
- **Açıklamalar.** Altı yöntem aynı görüntü ve eğitilmiş ağın seçtiği aynı hedef logit üzerinde hesaplanır. IG siyah referansla 24 aralıklı trapez integrali kullanır. Grad-CAM son evrişimde pozitif kanal bileşimini 28×28'e büyütür. Guided Backpropagation ReLU'da negatif geri sinyali kapılar.
- **Ölçüm.** Spearman sıra korelasyonu ham işaretli piksellerde ve mutlak değerlerinde ayrı hesaplanır. Sabit haritalarda tanımsız eşleşmeler ortalama dışında tutulur ve geçerli çift sayısı tabloda verilir.

## 4. Deneysel sonuçlar

| Koşul          |   Dönem |   Eğitim doğruluğu |   Özgün test doğruluğu |
|:---------------|--------:|-------------------:|-----------------------:|
| Gerçek etiket  |      12 |             1.0000 |                 0.9499 |
| Permüte etiket |      32 |             0.9856 |                 0.1025 |

![Eğitim eğrileri](sekiller/01-egitim.png)

Permüte etiketli ağ, verilen etiketlerde 98.6% eğitim doğruluğu ve özgün test etiketlerinde 10.2% doğruluk elde etti. Özgün test doğruluğu, on sınıflı görevdeki yaklaşık %10 şans düzeyiyle birlikte okunmalıdır. Kademeli testin ilk adımında işaretli ortalama korelasyon Gradyan için -0.006, Guided Backprop için 0.822 oldu. Son adımda aynı değerler sırasıyla -0.001 ve 0.503 olarak ölçüldü.

![Katmanlara göre benzerlik](sekiller/02-katman-benzerligi.png)

![Birinci MNIST örneği için haritalar](sekiller/03-haritalar.png)

![İkinci MNIST örneği için haritalar](sekiller/04-haritalar.png)

Görsellerde her harita için simetrik renk ölçeği kullanıldı. Paneller desen ve işaret karşılaştırmasına uygundur. Renk şiddeti paneller arasında mutlak büyüklük karşılaştırması için kullanılmaz.

### Permüte etiket müdahalesinde benzerlik

| Yöntem          | Ölçüt    |   Ortalama ρ |   Geçerli çift |
|:----------------|:---------|-------------:|---------------:|
| Girdi × gradyan | Mutlak   |        0.989 |             96 |
| Girdi × gradyan | İşaretli |        0.011 |             96 |
| Grad-CAM        | Mutlak   |        0.077 |             96 |
| Grad-CAM        | İşaretli |        0.077 |             96 |
| Gradyan         | Mutlak   |        0.073 |             96 |
| Gradyan         | İşaretli |        0.031 |             96 |
| Guided Backprop | Mutlak   |        0.547 |             96 |
| Guided Backprop | İşaretli |        0.650 |             96 |
| Guided Grad-CAM | Mutlak   |        0.186 |             96 |
| Guided Grad-CAM | İşaretli |        0.409 |             96 |
| IG              | Mutlak   |        0.989 |             96 |
| IG              | İşaretli |        0.021 |             96 |

![Etiket rastgeleleştirmesinde yöntemler](sekiller/05-etiket-benzerligi.png)

## 5. Makaleyle karşılaştırma ve sınırlılıklar

Makalede üst katmanlarda Guided Backprop haritalarının görece yüksek benzerlik gösterebildiği, alt katmanların da rastgeleleştirilmesiyle bozulabildiği bildirilir. Yukarıdaki ilk ve son adım ölçümleri bu ayrımı aynı MNIST görüntülerinde sınar. Girdi ile çarpılan yöntemlerde ortak sıfır pikselleri ve rakam silueti mutlak sıra korelasyonunu etkileyebilir. Atıf benzerliği modelin doğruluğundan ayrı değerlendirilir.

Bu çalışma TensorFlow yerine PyTorch kullanır. Eğitim altkümesi, Adam öğrenme hızı (0,001), 128'lik yığın, temiz model için 12 dönem ve permüte model için %98 çevrimiçi eğitim doğruluğunda erken durdurma veya en fazla 100 dönem yerel tercihlerdir. L2 cezası uygulanmadı. Özgün betikte doğrulama bölmesi ve 50'lik yığın vardır. Burada tek çalışma tohumu, altı atıf yöntemi ve iki Spearman ölçümü kullanıldı. HOG, SSIM, SmoothGrad ve VarGrad kapsam dışındadır. Bulgular bu CNN ve örneklemle sınırlıdır.
